In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import numpy as np
sys.path.append('../../../')   # Add parent directory to Python path
import pickle
from utils.preprocessing import *
from utils.segmentation import *
from utils.visualization import *

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical


np.random.seed(42)  # For reproducibility


# 1.1 Combine all datasets for training

In [5]:
# Load the curb data (which appears to be stored as a dictionary with scene_0 and scene_1)
with open('../../../data/Training/1s_30hz/curb_1s_combined_all.pkl', 'rb') as f:
    curb_data = pickle.load(f)
    data_curb_0 = curb_data['scene_0']
    data_curb_1 = curb_data['scene_1']
    
# Load the other surface types (these appear to be stored as arrays)
with open('../../../data/Training/1s_30hz/asphalt_1s_combined_all.pkl', 'rb') as f:
    data_asphalt = pickle.load(f)

with open('../../../data/Training/1s_30hz/cobblestone_1s_combined_all.pkl', 'rb') as f:
    data_cobblestone = pickle.load(f)

with open('../../../data/Training/1s_30hz/compactgravel_1s_combined_all.pkl', 'rb') as f:
    data_compact_gravel = pickle.load(f)

with open('../../../data/Training/1s_30hz/dirt_1s_combined_all.pkl', 'rb') as f:
    data_Dirt = pickle.load(f)

with open('../../../data/Training/1s_30hz/pavingstone_1s_combined_all.pkl', 'rb') as f:
    data_PavingStone = pickle.load(f)

# Print shapes to verify the data was loaded correctly
print("Curb (scene 0):", data_curb_0.shape)
print("Curb (scene 1):", data_curb_1.shape)
print("Asphalt:", data_asphalt.shape)
print("Cobblestone:", data_cobblestone.shape)
print("Compact Gravel:", data_compact_gravel.shape)
print("Dirt:", data_Dirt.shape)
print("Paving Stone:", data_PavingStone.shape)


Curb (scene 0): (12267, 30, 3)
Curb (scene 1): (617, 30, 3)
Asphalt: (704, 30, 3)
Cobblestone: (486, 30, 3)
Compact Gravel: (479, 30, 3)
Dirt: (577, 30, 3)
Paving Stone: (645, 30, 3)


In [ ]:
# real world data
with open('../../../data/Training/1s_30hz/real_world_1s_combined_3people_train.pkl', 'rb') as f:
    data_real_world = pickle.load(f)
    data_real_world_0 = data_real_world['scene_0']
    data_real_world_1 = data_real_world['scene_1']
print("Real World (scene 0):", data_real_world_0.shape)
print("Real World (scene 1):", data_real_world_1.shape)

## 1.1 Balanced binary dataset creation

In [ ]:
# Combine curb_1 data from both datasets
combined_curb_1 = np.concatenate([data_curb_1, data_real_world_1])

# Update the curb_1 data creation
curb_1_data = [(segment, "curb_1") for segment in combined_curb_1]
print(f"Total combined curb_1 samples: {len(curb_1_data)}")

# Get count of curb_1 samples to know how many we need from other classes
curb_1_count = len(curb_1_data)

# Create list of other datasets
other_datasets = [
    (data_curb_0, "curb_0"),
    (data_asphalt, "asphalt"),
    (data_cobblestone, "cobblestone"),
    (data_compact_gravel, "compact_gravel"),
    (data_Dirt, "dirt"),
    (data_PavingStone, "paving_stone"),
    (data_real_world_0, "real_world_0"),
]

# Calculate how many samples to take from each other class for even distribution
samples_per_class = curb_1_count // len(other_datasets)
print(f"Taking {samples_per_class} samples from each of the other 6 classes")

# Randomly select samples from other classes
other_class_data = []
for data, label in other_datasets:
    # Randomly select indices
    selected_indices = np.random.choice(len(data), samples_per_class, replace=False)
    # Add selected samples to other_class_data
    for idx in selected_indices:
        other_class_data.append((data[idx], "non_curb"))  # Label all other classes as "non_curb"

print(f"Total non_curb samples: {len(other_class_data)}")

# Combine curb_1 and other class data
combined_dataset = other_class_data + curb_1_data
print(f"Total binary dataset samples: {len(combined_dataset)}")

# Save the binary dataset
with open('../../data/1s_30hz/binary_dataset.pkl', 'wb') as f:
    pickle.dump(combined_dataset, f)



Total combined curb_1 samples: 881
Taking 125 samples from each of the other 6 classes
Total non_curb samples: 875
Total binary dataset samples: 1756


## 1.2 Unbalanced binary dataset creation - combine ALL data

### No real_world

In [ ]:


# Combine ALL curb_1 data (training datasets only)
combined_curb_1 = data_curb_1

# Combine ALL non-curb data (exclude real_world_0)
combined_non_curb = np.concatenate([
    data_curb_0,
    data_asphalt,
    data_cobblestone,
    data_compact_gravel,
    data_Dirt,
    data_PavingStone
])

print(f"Total curb_1 samples: {len(combined_curb_1)}")
print(f"Total non_curb samples: {len(combined_non_curb)}")

# Also create the combined format if you still need it
curb_1_data = [(segment, "curb_1") for segment in combined_curb_1]
non_curb_data = [(segment, "non_curb") for segment in combined_non_curb]
combined_dataset = non_curb_data + curb_1_data


Total curb_1 samples: 617
Total non_curb samples: 15158


### real world data

In [ ]:
combined_curb_1 = np.concatenate([data_curb_1, data_real_world_1])

# Combine ALL non-curb data from all datasets
combined_non_curb = np.concatenate([
    data_curb_0,
    data_asphalt,
    data_cobblestone,
    data_compact_gravel,
    data_Dirt,
    data_PavingStone,
    data_real_world_0
])

print(f"Total curb_1 samples: {len(combined_curb_1)}")
print(f"Total non_curb samples: {len(combined_non_curb)}")

# Also create the combined format if you still need it
curb_1_data = [(segment, "curb_1") for segment in combined_curb_1]
non_curb_data = [(segment, "non_curb") for segment in combined_non_curb]

combined_dataset = non_curb_data + curb_1_data

Total curb_1 samples: 881
Total non_curb samples: 26504
Saved separated classes with keys: 'non_curb', 'curb_1'


# 2. Train Test Spilt

## 2.1 Normal spilt

In [7]:
# 80% train, 20% test
train_set, test_set = train_test_split(combined_dataset, test_size=0.2, random_state=42, shuffle=True, stratify=[label for data, label in combined_dataset])
print(len(train_set), len(test_set))
# Separate sensor values and labels
X_train = [x for x, _ in train_set]
y_train = [label for _, label in train_set]
X_test = [x for x, _ in test_set]
y_test = [label for _, label in test_set]
print(f"Label train: {len(y_train)}, Label test: {len(y_test)}")

12620 3155
Label train: 12620, Label test: 3155


## 2.2 Leave one man out for testing

In [ ]:
# leave 3 men out for testing
with open('../../../data/Training/1s_30hz/real_world_1s_combined_3people_test.pkl', 'rb') as f:
    test_data = pickle.load(f)


In [ ]:
test_data_0 = test_data['scene_0']
test_data_1 = test_data['scene_1']
print(f"  test_data_0 shape: {test_data_0.shape}")
print(f"  test_data_1 shape: {test_data_1.shape}")



Test Data Shapes:
  test_data_0 shape: (10656, 30, 3)
  test_data_1 shape: (247, 30, 3)


# 3. Normalise dataset

In [9]:
X_train_normalized = normalize_3d_data(X_train)
X_test_normalized = normalize_3d_data(X_test)

# 4. Labels from string to integer

In [10]:
label_encoder = LabelEncoder()
y_train_int = label_encoder.fit_transform(y_train)
y_test_int = label_encoder.transform(y_test)

print("Classes:", label_encoder.classes_)
print("First 10 y_train_int:", y_train_int[:10])
print("First 10 y_test_int:", y_test_int[:10])
for idx, label in enumerate(label_encoder.classes_):
    print(f"{idx}: {label}")

Classes: ['curb_1' 'non_curb']
First 10 y_train_int: [1 1 1 1 1 1 1 1 1 1]
First 10 y_test_int: [1 0 1 1 1 1 1 1 1 1]
0: curb_1
1: non_curb


## 5: One-hot encode the labels

In [11]:
y_train_onehot = to_categorical(y_train_int)
y_test_onehot = to_categorical(y_test_int)

print(y_train_onehot.shape)
print(y_test_onehot.shape)

(12620, 2)
(3155, 2)


In [12]:
# Randomly select an index and check that the one-hot encoding matches the original label
r = np.random.randint(len(y_train_int))
assert y_train_onehot[r].argmax() == y_train_int[r]
r = np.random.randint(len(y_test_int))
assert y_test_onehot[r].argmax() == y_test_int[r]

## 6. Save train, test data and labels

In [13]:
# Save test data
with open('../../../data/Training/1s_30hz/2class_unbalanced/X_test_data.pkl', 'wb') as f:
    pickle.dump(X_test_normalized, f)
with open('../../../data/Training/1s_30hz/2class_unbalanced/y_test_onehot.pkl', 'wb') as f:
    pickle.dump(y_test_onehot, f)

## 6. Train, validation Spilt

In [14]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_normalized,
    y_train_onehot,
    test_size=0.2,           # 20% for validation
    random_state=42,
    shuffle=True
)

print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)

Train shape: (10096, 30, 3) (10096, 2)
Validation shape: (2524, 30, 3) (2524, 2)


In [15]:
# Save training and validation data
with open('../../../data/Training/1s_30hz/2class_unbalanced/X_train_normalized.pkl', 'wb') as f:
    pickle.dump(X_train, f)

with open('../../../data/Training/1s_30hz/2class_unbalanced/X_val_normalized.pkl', 'wb') as f:
    pickle.dump(X_val, f)

with open('../../../data/Training/1s_30hz/2class_unbalanced/y_train_onehot.pkl', 'wb') as f:
    pickle.dump(y_train, f)

with open('../../../data/Training/1s_30hz/2class_unbalanced/y_val_onehot.pkl', 'wb') as f:
    pickle.dump(y_val, f)